# Importing modules and functions

In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Descriptors import MoleculeDescriptors
import chembl_structure_pipeline
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.model_selection import permutation_test_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict
from sklearn import metrics
from sklearn.metrics import pairwise_distances
import joblib
import pickle
from molvs import standardize_smiles
from numpy import savetxt
from padelpy import from_sdf
from IPython.display import HTML
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor
import warnings
from sklearn.neighbors import KNeighborsRegressor
warnings.filterwarnings('ignore')

[09:50:42] Initializing Normalizer


# Data entry and curation work set

In [2]:
uploaded_file_ws="datasets/HDAC8_work.sdf"
supplier_ws = Chem.ForwardSDMolSupplier(uploaded_file_ws,sanitize=False)
failed_mols_ws = []
all_mols_ws =[]
wrong_structure_ws=[]
wrong_smiles_ws=[]
y_tr = []
y_bad_index=[]

for i, m in enumerate(supplier_ws):
    structure = Chem.Mol(m)
    all_mols_ws.append(structure)
    y_tr.append(m.GetProp("pchembl_value_mean"))
    try:
        Chem.SanitizeMol(structure)
    except:
        failed_mols_ws.append(m)
        wrong_smiles_ws.append(Chem.MolToSmiles(m))
        wrong_structure_ws.append(str(i+1))
        y_bad_index.append(i)
print('Original data: ', len(all_mols_ws), 'molecules')
print('Failed data: ', len(failed_mols_ws), 'molecules')
number_ws =[]
for i in range(len(failed_mols_ws)):
        number_ws.append(str(i+1))
bad_molecules_ws = pd.DataFrame({'No. failed molecule in original set': wrong_structure_ws, 'SMILES of wrong structure: ': wrong_smiles_ws, 'No.': number_ws}, index=None)
bad_molecules_ws = bad_molecules_ws.set_index('No.')
bad_molecules_ws

Original data:  2210 molecules
Failed data:  0 molecules


,No. failed molecule in original set,SMILES of wrong structure:
No.,,


deleting activity values for substances with incorrect structure

In [3]:
y_tr[:] = [x for i,x in enumerate(y_tr) if i not in y_bad_index]

In [4]:
len(y_tr)

2210

# Standardization SDF file for work set

In [5]:
all_mols_ws[:] = [x for i,x in enumerate(all_mols_ws) if i not in y_bad_index] 
records = []
for i in range(len(all_mols_ws)):
    record = Chem.MolToSmiles(all_mols_ws[i])
    records.append(record)

moldf_ws = []
for i,record in enumerate(records):
    standard_record = standardize_smiles(record)
    m = Chem.MolFromSmiles(standard_record)
    moldf_ws.append(m)
    
print('Kept data: ', len(moldf_ws), 'molecules')

Kept data:  2210 molecules


In [6]:
moldf_ws=pd.DataFrame(moldf_ws, columns=['Mol'])
moldf_ws

,Mol
0,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
1,<rdkit.Chem.rdchem.Mol object at 0x00000133000...
2,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
3,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
4,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
...,...
2205,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
2206,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
2207,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
2208,<rdkit.Chem.rdchem.Mol object at 0x00000133011...


# Data entry and curation test set

In [7]:
uploaded_file_ts="datasets/HDAC8_test.sdf"
supplier_ts = Chem.ForwardSDMolSupplier(uploaded_file_ts,sanitize=False)
failed_mols_ts = []
all_mols_ts =[]
wrong_structure_ts=[]
wrong_smiles_ts=[]
y_ts = []
y_bad_index=[]
for i, m in enumerate(supplier_ts):
    structure = Chem.Mol(m)
    all_mols_ts.append(structure)
    y_ts.append(m.GetProp("pchembl_value_mean"))
    try:
        Chem.SanitizeMol(structure)
    except:
        failed_mols_ts.append(m)
        wrong_smiles_ts.append(Chem.MolToSmiles(m))
        wrong_structure_ts.append(str(i+1))
        y_bad_index.append(i)
print('Original data: ', len(all_mols_ts), 'molecules')
print('Failed data: ', len(failed_mols_ts), 'molecules')
number_ts =[]
for i in range(len(failed_mols_ts)):
        number_ts.append(str(i+1))
bad_molecules_ts = pd.DataFrame({'No. failed molecule in original set': wrong_structure_ts, 'SMILES of wrong structure: ': wrong_smiles_ts, 'No.': number_ts}, index=None)
bad_molecules_ts = bad_molecules_ts.set_index('No.')
bad_molecules_ts

Original data:  553 molecules
Failed data:  0 molecules


,No. failed molecule in original set,SMILES of wrong structure:
No.,,


deleting activity values for substances with incorrect structure

In [8]:
y_ts[:] = [x for i,x in enumerate(y_ts) if i not in y_bad_index]

In [9]:
len(y_ts)

553

# Standardization SDF file for test set

In [10]:
all_mols_ts[:] = [x for i,x in enumerate(all_mols_ts) if i not in y_bad_index] 
records = []
for i in range(len(all_mols_ts)):
    record = Chem.MolToSmiles(all_mols_ts[i])
    records.append(record)

moldf_ts = []
for i,record in enumerate(records):
    standard_record = standardize_smiles(record)
    m = Chem.MolFromSmiles(standard_record)
    moldf_ts.append(m)
    
print('Kept data: ', len(moldf_ts), 'molecules')

Kept data:  553 molecules


In [11]:
moldf_ts=pd.DataFrame(moldf_ts, columns=['Mol'])
moldf_ts

,Mol
0,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
1,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
2,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
3,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
4,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
...,...
548,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
549,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
550,<rdkit.Chem.rdchem.Mol object at 0x00000133011...
551,<rdkit.Chem.rdchem.Mol object at 0x00000133011...


# Calculation ECFP4 for work set

In [12]:
import warnings
from rdkit import RDLogger

# Suppress RDKit warnings (including deprecation warnings)
RDLogger.DisableLog('rdApp.*')

def calcfp(mol, funcFPInfo=dict(radius=2, nBits=1024, useFeatures=False, useChirality=False)):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, **funcFPInfo)
    fp = pd.Series(np.asarray(fp))
    fp = fp.add_prefix('Bit_')
    return fp

In [13]:
# Training set
desc_ws = moldf_ws.Mol.apply(calcfp)
desc_ws

,Bit_0,Bit_1,Bit_2,Bit_3,Bit_4,Bit_5,Bit_6,Bit_7,Bit_8,Bit_9,...,Bit_1014,Bit_1015,Bit_1016,Bit_1017,Bit_1018,Bit_1019,Bit_1020,Bit_1021,Bit_1022,Bit_1023
0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2206,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,0,0,0,0
2207,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,0,0,0,0
2208,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
savetxt('models/ECFP4/x_tr_ECFP4.csv', desc_ws, delimiter=',')

In [15]:
y_tr = np.array(y_tr, dtype=np.float32)
len(y_tr)

2210

# Calculation ECFP4 for test set

In [16]:
desc_ts = moldf_ts.Mol.apply(calcfp)
desc_ts

,Bit_0,Bit_1,Bit_2,Bit_3,Bit_4,Bit_5,Bit_6,Bit_7,Bit_8,Bit_9,...,Bit_1014,Bit_1015,Bit_1016,Bit_1017,Bit_1018,Bit_1019,Bit_1020,Bit_1021,Bit_1022,Bit_1023
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
549,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
550,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
551,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [37]:
y_ts = np.array(y_ts, dtype=np.float32)
len(y_ts)

553

# BASELINE

 ## GradientBoostingRegressor model building and validation

In [39]:
seed = 42

In [40]:
cv=KFold(n_splits=5, random_state=seed, shuffle=True)

In [27]:
%%time
model = CatBoostRegressor()
parameters = {'depth' : [6,8,10],
              'learning_rate' : [0.01, 0.05, 0.1],
              'iterations'    : [100,500, 1000]
              }

grid = GridSearchCV(estimator=model, param_grid = parameters, n_jobs=-1, cv = cv)
grid.fit(desc_ws, y_tr, verbose=False)

CPU times: total: 9min 54s
Wall time: 15min 41s


,estimator,CatBoostRegre...nction='RMSE')
,param_grid,"{'depth': [6, 8, ...], 'iterations': [100, 500, ...], 'learning_rate': [0.01, 0.05, ...]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False


In [28]:
best_CatBR = grid.best_estimator_

In [29]:
grid.best_params_

{'depth': 10, 'iterations': 1000, 'learning_rate': 0.05}

In [46]:
y_pred_ws_GBR = best_CatBR.predict(desc_ws)

In [47]:
R2_WS = round(r2_score(y_tr, y_pred_ws_GBR), 2)
R2_WS

0.95

In [48]:
RMSE_WS=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_ws_GBR)), 2))
RMSE_WS

0.19

In [49]:
params={'verbose': False}

In [50]:
%%time
y_pred_CV_CatBR = cross_val_predict(best_CatBR, desc_ws, y_tr, cv=cv, params=params)

CPU times: total: 39min 53s
Wall time: 3min 35s


In [51]:
Q2_CV = round(r2_score(y_tr, y_pred_CV_CatBR), 2)
Q2_CV

0.58

In [52]:
RMSE_CV = float(round(np.sqrt(mean_squared_error(y_tr, y_pred_CV_CatBR)), 2))
RMSE_CV

0.55

In [53]:
y_pred_GBR = best_CatBR.predict(desc_ts)

In [54]:
Q2_TS = round(r2_score(y_ts, y_pred_GBR), 2)
Q2_TS

0.58

In [55]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts, y_pred_GBR)), 2))
RMSE_TS

0.55

save the model to disk

In [31]:
pickle.dump(best_CatBR, open('models/ECFP4/CatBoost_ECFP4.pkl', 'wb'))

load the model from disk

In [45]:
best_CatBR = pickle.load(open('models/ECFP4/CatBoost_ECFP4.pkl', 'rb'))

# Y-randomization

In [81]:
params={'verbose': False}

In [82]:
permutations = 50
score, permutation_scores, pvalue = permutation_test_score(best_CatBR, desc_ws, y_tr,
                                                           cv=cv, scoring='r2',
                                                           n_permutations=permutations,
                                                           n_jobs=-1,
                                                           params=params,
                                                           random_state=seed)
print('True score = ', score.round(2),
      '\nY-randomization = ', np.mean(permutation_scores).round(2),
      '\np-value = ', pvalue.round(4))

True score =  0.57 
Y-randomization =  -0.21 
p-value =  0.0196


# Estimating applicability domain. Method - Euclidian distances, K=1

In [56]:
neighbors_k= pairwise_distances(desc_ws, n_jobs=-1)
neighbors_k.sort(0)

In [57]:
df_tr=pd.DataFrame(neighbors_k)
df_tr

,0,1,2,3,4,5,6,7,8,9,...,2200,2201,2202,2203,2204,2205,2206,2207,2208,2209
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,5.916080,3.162278,3.872983,0.000000,2.645751,3.162278,3.162278,0.000000,3.741657,4.123106,...,3.872983,3.464102,7.000000,3.464102,4.898979,6.928203,0.000000,0.000000,4.795832,3.316625
2,6.000000,3.741657,5.196152,2.645751,2.645751,4.242641,3.162278,2.645751,5.291503,4.795832,...,4.472136,4.242641,7.071068,3.605551,6.708204,7.000000,3.000000,3.000000,5.291503,3.605551
3,6.403124,4.000000,5.196152,2.828427,3.000000,4.472136,3.162278,2.828427,5.291503,5.830952,...,5.000000,4.795832,7.211103,3.872983,7.211103,7.000000,3.162278,3.162278,5.567764,3.872983
4,6.480741,4.123106,5.196152,3.162278,3.316625,4.795832,3.316625,3.162278,5.385165,5.830952,...,5.656854,5.000000,7.483315,4.000000,7.211103,7.000000,3.316625,3.316625,5.567764,4.242641
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,11.180340,10.392305,11.090537,11.090537,11.224972,11.224972,11.357817,11.090537,11.618950,11.180340,...,11.532563,10.488088,11.357817,10.723805,10.908712,10.862780,10.723805,10.723805,11.180340,11.045361
2206,11.180340,10.440307,11.180340,11.180340,11.313708,11.224972,11.445523,11.180340,11.618950,11.180340,...,11.532563,10.535654,11.357817,10.723805,10.908712,10.954451,10.723805,10.723805,11.180340,11.090537
2207,11.180340,10.440307,11.180340,11.180340,11.313708,11.224972,11.445523,11.180340,11.618950,11.180340,...,11.575837,10.535654,11.357817,10.770330,11.045361,10.954451,10.723805,10.723805,11.180340,11.090537
2208,11.180340,10.440307,11.489125,11.224972,11.357817,11.269428,11.489125,11.224972,11.618950,11.224972,...,11.618950,10.535654,11.401754,10.816654,11.135529,10.954451,10.723805,10.723805,11.224972,11.224972


In [58]:
similarity= neighbors_k

In [59]:
Dmean=np.mean(similarity[1,:])

In [60]:
round(Dmean, 2)

np.float64(3.37)

In [61]:
std=np.std(similarity[1,:])

In [62]:
round(std, 2)

np.float64(1.53)

In [63]:
model_AD_limit=Dmean+std*0.5
print(np.round(model_AD_limit, 2))

4.13


In [64]:
neighbors_k_ts= pairwise_distances(desc_ws,Y=desc_ts, n_jobs=-1)
neighbors_k_ts.sort(0)

In [65]:
x_ts_AD=pd.DataFrame(neighbors_k_ts)
x_ts_AD

,0,1,2,3,4,5,6,7,8,9,...,543,544,545,546,547,548,549,550,551,552
0,5.000000,5.291503,3.605551,4.582576,3.741657,4.690416,3.605551,3.316625,4.242641,3.464102,...,5.099020,3.162278,4.242641,2.449490,2.645751,3.741657,3.162278,0.000000,1.732051,4.123106
1,5.099020,5.744563,4.472136,5.744563,3.741657,4.690416,3.605551,4.472136,4.472136,4.000000,...,5.196152,3.741657,4.242641,2.449490,2.828427,4.242641,3.162278,0.000000,3.464102,4.472136
2,5.099020,6.000000,4.582576,5.744563,4.242641,4.898979,3.872983,4.690416,5.196152,4.123106,...,5.196152,4.000000,4.690416,3.316625,2.828427,4.242641,3.464102,1.414214,3.464102,4.472136
3,5.196152,6.082763,5.000000,5.744563,4.358899,5.000000,4.242641,5.291503,5.196152,4.242641,...,5.385165,4.123106,4.690416,3.741657,3.162278,4.472136,3.872983,3.000000,3.872983,4.472136
4,5.291503,6.164414,5.099020,5.830952,4.358899,5.196152,4.358899,5.291503,5.291503,4.242641,...,5.656854,4.472136,4.795832,3.741657,3.464102,4.690416,4.123106,3.605551,3.872983,4.795832
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,11.357817,11.045361,11.045361,11.135529,11.000000,11.269428,10.862780,11.661904,11.224972,11.445523,...,11.532563,10.440307,10.535654,10.148892,10.440307,10.344080,10.677078,10.488088,10.246951,11.575837
2206,11.357817,11.045361,11.090537,11.180340,11.090537,11.269428,10.862780,11.661904,11.224972,11.445523,...,11.532563,10.440307,10.535654,10.198039,10.488088,10.344080,10.677078,10.488088,10.246951,11.618950
2207,11.357817,11.090537,11.090537,11.180340,11.090537,11.269428,10.862780,11.661904,11.224972,11.445523,...,11.575837,10.488088,10.583005,10.198039,10.488088,10.392305,10.723805,10.535654,10.246951,11.618950
2208,11.445523,11.090537,11.135529,11.180340,11.313708,11.269428,10.908712,11.874342,11.704700,11.747340,...,11.575837,10.488088,10.583005,10.246951,10.488088,10.440307,10.770330,10.583005,10.246951,11.747340


In [66]:
similarity_ts= neighbors_k_ts
cpd_AD=similarity_ts[0,:]
cpd_value = np.round(cpd_AD, 3)
print(cpd_value)

[5.    5.292 3.606 4.583 3.742 4.69  3.606 3.317 4.243 3.464 5.099 3.162
 2.828 3.    4.    3.162 3.464 3.464 2.449 4.796 0.    1.414 3.317 4.472
 3.317 1.732 3.606 3.873 3.464 3.464 3.464 4.583 3.    4.    3.    4.359
 2.646 3.873 5.    4.69  5.099 5.657 2.646 0.    3.742 2.828 3.464 3.742
 2.449 4.    3.162 4.472 3.317 2.646 1.414 3.742 4.    5.385 0.    0.
 3.162 5.477 3.873 3.162 3.162 4.    3.    4.899 3.162 2.    4.243 3.873
 0.    4.    2.828 0.    6.    3.162 3.873 2.646 3.    1.    4.    3.464
 4.359 3.742 4.    0.    3.873 4.583 3.    3.464 4.243 3.    4.    3.873
 4.899 4.243 0.    3.742 3.873 3.162 3.    2.828 3.742 3.742 3.742 4.359
 1.    1.732 3.464 4.243 3.742 4.359 0.    4.359 3.    3.162 3.317 3.606
 3.742 0.    2.449 5.196 0.    2.828 1.    3.162 3.742 4.359 4.583 4.69
 4.69  6.708 3.742 3.742 1.414 3.606 1.    3.873 4.359 2.    3.742 3.317
 3.317 1.    3.    3.873 3.    1.    0.    3.317 1.    3.317 3.162 1.
 8.246 4.69  4.69  3.162 3.162 1.414 4.359 4.472 3.873 3. 

In [67]:
cpd_AD = np.where(cpd_value <= model_AD_limit, True, False)
print(cpd_AD)

[False False  True False  True False  True  True False  True False  True
  True  True  True  True  True  True  True False  True  True  True False
  True  True  True  True  True  True  True False  True  True  True False
  True  True False False False False  True  True  True  True  True  True
  True  True  True False  True  True  True  True  True False  True  True
  True False  True  True  True  True  True False  True  True False  True
  True  True  True  True False  True  True  True  True  True  True  True
 False  True  True  True  True False  True  True False  True  True  True
 False False  True  True  True  True  True  True  True  True  True False
  True  True  True False  True False  True False  True  True  True  True
  True  True  True False  True  True  True  True  True False False False
 False False  True  True  True  True  True  True False  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
 False False False  True  True  True False False  T

In [68]:
print("Coverage = ", round(sum(cpd_AD) / len(cpd_AD),2))

Coverage =  0.77


In [69]:
print("Indices of substances included in AD = ", np.where(cpd_AD != 0)[0])

Indices of substances included in AD =  [  2   4   6   7   9  11  12  13  14  15  16  17  18  20  21  22  24  25
  26  27  28  29  30  32  33  34  36  37  42  43  44  45  46  47  48  49
  50  52  53  54  55  56  58  59  60  62  63  64  65  66  68  69  71  72
  73  74  75  77  78  79  80  81  82  83  85  86  87  88  90  91  93  94
  95  98  99 100 101 102 103 104 105 106 108 109 110 112 114 116 117 118
 119 120 121 122 124 125 126 127 128 134 135 136 137 138 139 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 159 160 161 164 165 166
 169 170 171 172 173 174 175 176 178 179 180 181 182 183 184 186 187 191
 193 194 195 199 202 203 205 206 208 209 210 211 213 214 215 216 217 218
 221 222 224 225 226 228 229 230 231 233 234 235 236 237 238 239 240 241
 242 243 244 245 246 247 248 249 250 251 252 253 255 256 258 259 261 262
 263 265 266 267 268 269 270 273 274 276 277 278 279 282 283 284 286 287
 288 289 290 291 293 295 296 298 299 300 301 304 305 307 308 309 310 311
 312 313 31

In [70]:
out_Ad=list(np.where(cpd_AD == 0)[0])

# Prediction only for molecules included in  AD

In [71]:
y_pred_GBR_ad=list(y_pred_GBR)

In [72]:
y_pred_GBR_ad[:] = [x for i,x in enumerate(y_pred_GBR_ad) if i not in out_Ad]

In [73]:
len(y_pred_GBR_ad)

424

In [74]:
y_ts_ad=list(y_ts)

In [75]:
y_ts_ad[:] = [x for i,x in enumerate(y_ts_ad) if i not in out_Ad]

In [76]:
len(y_ts_ad)

424

In [77]:
Q2_TS = round(r2_score(y_ts_ad, y_pred_GBR_ad), 2)
Q2_TS

0.63

In [78]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts_ad, y_pred_GBR_ad)), 2))
RMSE_TS

0.52

# SVM model building and validation

In [106]:
from sklearn.svm import SVR

In [107]:
param_grid = {"C": [10 ** i for i in range(0, 5)],
              "gamma": [10 ** i for i in range(-6, 0)]}

In [108]:
seed = 42
cv=KFold(n_splits=5, random_state=seed, shuffle=True)

In [109]:
svm = GridSearchCV(SVR(C=1.0, epsilon=0.2), param_grid, n_jobs=-1, cv=cv, verbose=1)

In [110]:
svm.fit(desc_ws, y_tr)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,SVR(epsilon=0.2)
,param_grid,"{'C': [1, 10, ...], 'gamma': [1e-06, 1e-05, ...]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,kernel,'rbf'


In [111]:
svm.best_params_
best_svm = svm.best_estimator_

In [112]:
svm.best_params_

{'C': 10, 'gamma': 0.01}

In [81]:
y_pred_ws_SVM = best_svm.predict(desc_ws)

In [82]:
R2_WS = round(r2_score(y_tr, y_pred_ws_SVM), 2)
R2_WS

0.93

In [83]:
RMSE_WS=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_ws_SVM)), 2))
RMSE_WS

0.23

In [84]:
y_pred_CV_svm = cross_val_predict(best_svm, desc_ws, y_tr, cv=cv)

In [85]:
Q2_CV = round(r2_score(y_tr, y_pred_CV_svm), 2)
Q2_CV

0.53

In [86]:
RMSE_CV=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_CV_svm)), 2))
RMSE_CV

0.58

#  Prediction for test set's molecules

In [87]:
x_ts = np.array(desc_ts, dtype=np.float32)
y_ts = np.array(y_ts, dtype=np.float32)

In [88]:
y_pred_svm = best_svm.predict(x_ts)

In [89]:
Q2_TS = round(r2_score(y_ts, y_pred_svm), 2)
Q2_TS

0.53

In [90]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts, y_pred_svm)), 2))
RMSE_TS

0.59

save the model to disk

In [120]:
pickle.dump(best_svm, open('models/MorganFingerprint/Morgan_SVM.pkl', 'wb'))

load the model from disk

In [80]:
best_svm = pickle.load(open('models/ECFP4/SVM_ECFP4.pkl', 'rb'))

# Estimating applicability domain. Method - Euclidian distances, K=1

In [91]:
neighbors_k= pairwise_distances(desc_ws, n_jobs=-1)
neighbors_k.sort(0)

In [92]:
df_tr=pd.DataFrame(neighbors_k)
df_tr

,0,1,2,3,4,5,6,7,8,9,...,2200,2201,2202,2203,2204,2205,2206,2207,2208,2209
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,5.916080,3.162278,3.872983,0.000000,2.645751,3.162278,3.162278,0.000000,3.741657,4.123106,...,3.872983,3.464102,7.000000,3.464102,4.898979,6.928203,0.000000,0.000000,4.795832,3.316625
2,6.000000,3.741657,5.196152,2.645751,2.645751,4.242641,3.162278,2.645751,5.291503,4.795832,...,4.472136,4.242641,7.071068,3.605551,6.708204,7.000000,3.000000,3.000000,5.291503,3.605551
3,6.403124,4.000000,5.196152,2.828427,3.000000,4.472136,3.162278,2.828427,5.291503,5.830952,...,5.000000,4.795832,7.211103,3.872983,7.211103,7.000000,3.162278,3.162278,5.567764,3.872983
4,6.480741,4.123106,5.196152,3.162278,3.316625,4.795832,3.316625,3.162278,5.385165,5.830952,...,5.656854,5.000000,7.483315,4.000000,7.211103,7.000000,3.316625,3.316625,5.567764,4.242641
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,11.180340,10.392305,11.090537,11.090537,11.224972,11.224972,11.357817,11.090537,11.618950,11.180340,...,11.532563,10.488088,11.357817,10.723805,10.908712,10.862780,10.723805,10.723805,11.180340,11.045361
2206,11.180340,10.440307,11.180340,11.180340,11.313708,11.224972,11.445523,11.180340,11.618950,11.180340,...,11.532563,10.535654,11.357817,10.723805,10.908712,10.954451,10.723805,10.723805,11.180340,11.090537
2207,11.180340,10.440307,11.180340,11.180340,11.313708,11.224972,11.445523,11.180340,11.618950,11.180340,...,11.575837,10.535654,11.357817,10.770330,11.045361,10.954451,10.723805,10.723805,11.180340,11.090537
2208,11.180340,10.440307,11.489125,11.224972,11.357817,11.269428,11.489125,11.224972,11.618950,11.224972,...,11.618950,10.535654,11.401754,10.816654,11.135529,10.954451,10.723805,10.723805,11.224972,11.224972


In [93]:
similarity= neighbors_k

In [94]:
Dmean=np.mean(similarity[1,:])

In [95]:
round(Dmean, 2)

np.float64(3.37)

In [96]:
std=np.std(similarity[1,:])

In [97]:
round(std, 2)

np.float64(1.53)

In [98]:
model_AD_limit=Dmean+std*0.5
print(np.round(model_AD_limit, 2))

4.13


In [99]:
neighbors_k_ts= pairwise_distances(desc_ws,Y=x_ts, n_jobs=-1)
neighbors_k_ts.sort(0)

In [100]:
x_ts_AD=pd.DataFrame(neighbors_k_ts)
x_ts_AD

,0,1,2,3,4,5,6,7,8,9,...,543,544,545,546,547,548,549,550,551,552
0,5.000000,5.291503,3.605551,4.582576,3.741657,4.690416,3.605551,3.316625,4.242641,3.464102,...,5.099020,3.162278,4.242641,2.449490,2.645751,3.741657,3.162278,0.000000,1.732051,4.123106
1,5.099020,5.744563,4.472136,5.744563,3.741657,4.690416,3.605551,4.472136,4.472136,4.000000,...,5.196152,3.741657,4.242641,2.449490,2.828427,4.242641,3.162278,0.000000,3.464102,4.472136
2,5.099020,6.000000,4.582576,5.744563,4.242641,4.898979,3.872983,4.690416,5.196152,4.123106,...,5.196152,4.000000,4.690416,3.316625,2.828427,4.242641,3.464102,1.414214,3.464102,4.472136
3,5.196152,6.082763,5.000000,5.744563,4.358899,5.000000,4.242641,5.291503,5.196152,4.242641,...,5.385165,4.123106,4.690416,3.741657,3.162278,4.472136,3.872983,3.000000,3.872983,4.472136
4,5.291503,6.164414,5.099020,5.830952,4.358899,5.196152,4.358899,5.291503,5.291503,4.242641,...,5.656854,4.472136,4.795832,3.741657,3.464102,4.690416,4.123106,3.605551,3.872983,4.795832
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,11.357817,11.045361,11.045361,11.135529,11.000000,11.269428,10.862780,11.661904,11.224972,11.445523,...,11.532563,10.440307,10.535654,10.148892,10.440307,10.344080,10.677078,10.488088,10.246951,11.575837
2206,11.357817,11.045361,11.090537,11.180340,11.090537,11.269428,10.862780,11.661904,11.224972,11.445523,...,11.532563,10.440307,10.535654,10.198039,10.488088,10.344080,10.677078,10.488088,10.246951,11.618950
2207,11.357817,11.090537,11.090537,11.180340,11.090537,11.269428,10.862780,11.661904,11.224972,11.445523,...,11.575837,10.488088,10.583005,10.198039,10.488088,10.392305,10.723805,10.535654,10.246951,11.618950
2208,11.445523,11.090537,11.135529,11.180340,11.313708,11.269428,10.908712,11.874342,11.704700,11.747340,...,11.575837,10.488088,10.583005,10.246951,10.488088,10.440307,10.770330,10.583005,10.246951,11.747340


In [101]:
similarity_ts= neighbors_k_ts
cpd_AD=similarity_ts[0,:]
cpd_value = np.round(cpd_AD, 3)
print(cpd_value)

[5.    5.292 3.606 4.583 3.742 4.69  3.606 3.317 4.243 3.464 5.099 3.162
 2.828 3.    4.    3.162 3.464 3.464 2.449 4.796 0.    1.414 3.317 4.472
 3.317 1.732 3.606 3.873 3.464 3.464 3.464 4.583 3.    4.    3.    4.359
 2.646 3.873 5.    4.69  5.099 5.657 2.646 0.    3.742 2.828 3.464 3.742
 2.449 4.    3.162 4.472 3.317 2.646 1.414 3.742 4.    5.385 0.    0.
 3.162 5.477 3.873 3.162 3.162 4.    3.    4.899 3.162 2.    4.243 3.873
 0.    4.    2.828 0.    6.    3.162 3.873 2.646 3.    1.    4.    3.464
 4.359 3.742 4.    0.    3.873 4.583 3.    3.464 4.243 3.    4.    3.873
 4.899 4.243 0.    3.742 3.873 3.162 3.    2.828 3.742 3.742 3.742 4.359
 1.    1.732 3.464 4.243 3.742 4.359 0.    4.359 3.    3.162 3.317 3.606
 3.742 0.    2.449 5.196 0.    2.828 1.    3.162 3.742 4.359 4.583 4.69
 4.69  6.708 3.742 3.742 1.414 3.606 1.    3.873 4.359 2.    3.742 3.317
 3.317 1.    3.    3.873 3.    1.    0.    3.317 1.    3.317 3.162 1.
 8.246 4.69  4.69  3.162 3.162 1.414 4.359 4.472 3.873 3. 

In [102]:
cpd_AD = np.where(cpd_value <= model_AD_limit, True, False)
print(cpd_AD)

[False False  True False  True False  True  True False  True False  True
  True  True  True  True  True  True  True False  True  True  True False
  True  True  True  True  True  True  True False  True  True  True False
  True  True False False False False  True  True  True  True  True  True
  True  True  True False  True  True  True  True  True False  True  True
  True False  True  True  True  True  True False  True  True False  True
  True  True  True  True False  True  True  True  True  True  True  True
 False  True  True  True  True False  True  True False  True  True  True
 False False  True  True  True  True  True  True  True  True  True False
  True  True  True False  True False  True False  True  True  True  True
  True  True  True False  True  True  True  True  True False False False
 False False  True  True  True  True  True  True False  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
 False False False  True  True  True False False  T

In [103]:
print("Coverage = ", round(sum(cpd_AD) / len(cpd_AD), 2))

Coverage =  0.77


In [104]:
print("Indices of substances included in AD = ", np.where(cpd_AD != 0)[0])

Indices of substances included in AD =  [  2   4   6   7   9  11  12  13  14  15  16  17  18  20  21  22  24  25
  26  27  28  29  30  32  33  34  36  37  42  43  44  45  46  47  48  49
  50  52  53  54  55  56  58  59  60  62  63  64  65  66  68  69  71  72
  73  74  75  77  78  79  80  81  82  83  85  86  87  88  90  91  93  94
  95  98  99 100 101 102 103 104 105 106 108 109 110 112 114 116 117 118
 119 120 121 122 124 125 126 127 128 134 135 136 137 138 139 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 159 160 161 164 165 166
 169 170 171 172 173 174 175 176 178 179 180 181 182 183 184 186 187 191
 193 194 195 199 202 203 205 206 208 209 210 211 213 214 215 216 217 218
 221 222 224 225 226 228 229 230 231 233 234 235 236 237 238 239 240 241
 242 243 244 245 246 247 248 249 250 251 252 253 255 256 258 259 261 262
 263 265 266 267 268 269 270 273 274 276 277 278 279 282 283 284 286 287
 288 289 290 291 293 295 296 298 299 300 301 304 305 307 308 309 310 311
 312 313 31

In [105]:
out_Ad=list(np.where(cpd_AD == 0)[0])

# Prediction only for molecules included in  AD

In [106]:
y_pred_svm_ad=list(y_pred_svm)

In [107]:
y_pred_svm_ad[:] = [x for i,x in enumerate(y_pred_svm_ad) if i not in out_Ad]

In [108]:
len(y_pred_svm_ad)

424

In [109]:
y_ts_ad=list(y_ts)

In [110]:
y_ts_ad[:] = [x for i,x in enumerate(y_ts_ad) if i not in out_Ad]

In [111]:
len(y_ts_ad)

424

In [112]:
Q2_TS = round(r2_score(y_ts_ad, y_pred_svm_ad), 2)
Q2_TS

0.59

In [113]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts_ad, y_pred_svm_ad)), 2))
RMSE_TS

0.54

# Multi-layer Perceptron regressor

In [41]:
from sklearn.neural_network import MLPRegressor

In [42]:
param_grid ={"hidden_layer_sizes": [(400, 300, 200, 100),(100, 100, 100), (10, 10, 10),(50,)], "activation": ["tanh", "relu"], "solver": ["lbfgs", "sgd", "adam"], "alpha": [0.00005,0.0005], 'max_iter': [1000, 2000]}

In [43]:
m = GridSearchCV(MLPRegressor(), param_grid, n_jobs=-1, cv=cv, verbose=1)

In [44]:
m.fit(desc_ws, y_tr)

Fitting 5 folds for each of 96 candidates, totalling 480 fits


,estimator,MLPRegressor()
,param_grid,"{'activation': ['tanh', 'relu'], 'alpha': [5e-05, 0.0005], 'hidden_layer_sizes': [(400, ...), (100, ...), ...], 'max_iter': [1000, 2000], ...}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'squared_error'


In [45]:
best_MLPR = m.best_estimator_

In [46]:
y_pred_ws_best_MLPR = best_MLPR.predict(desc_ws)

In [47]:
R2_WS = round(r2_score(y_tr, y_pred_ws_best_MLPR), 2)
R2_WS

0.99

In [49]:
RMSE_WS=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_ws_best_MLPR)), 2))
RMSE_WS

0.1

In [50]:
y_pred_CV_MLPR = cross_val_predict(best_MLPR, desc_ws, y_tr, cv=cv)

In [51]:
Q2_CV = round(r2_score(y_tr, y_pred_CV_MLPR), 2)
Q2_CV

0.49

In [52]:
RMSE_CV=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_CV_MLPR)), 2))
RMSE_CV

0.61

# Prediction for test set's molecules

In [53]:
y_pred_MLPR = best_MLPR.predict(desc_ts)

In [54]:
Q2_TS = round(r2_score(y_ts, y_pred_MLPR), 2)
Q2_TS

0.47

In [55]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts, y_pred_MLPR)), 2))
RMSE_TS

0.62

save the model to disk

In [56]:
pickle.dump(best_MLPR, open('models/ECFP4/MLPR_ECFP4.pkl', 'wb'))

load the model from disk

In [123]:
best_MLPR = pickle.load(open('models/ECFP4/MLPR_ECFP4.pkl', 'rb'))

#  Estimating applicability domain. Method - Euclidian distances, K=1

In [57]:
neighbors_k= pairwise_distances(desc_ws, n_jobs=-1)
neighbors_k.sort(0)

In [58]:
df_tr=pd.DataFrame(neighbors_k)
df_tr

,0,1,2,3,4,5,6,7,8,9,...,2200,2201,2202,2203,2204,2205,2206,2207,2208,2209
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,5.916080,3.162278,3.872983,0.000000,2.645751,3.162278,3.162278,0.000000,3.741657,4.123106,...,3.872983,3.464102,7.000000,3.464102,4.898979,6.928203,0.000000,0.000000,4.795832,3.316625
2,6.000000,3.741657,5.196152,2.645751,2.645751,4.242641,3.162278,2.645751,5.291503,4.795832,...,4.472136,4.242641,7.071068,3.605551,6.708204,7.000000,3.000000,3.000000,5.291503,3.605551
3,6.403124,4.000000,5.196152,2.828427,3.000000,4.472136,3.162278,2.828427,5.291503,5.830952,...,5.000000,4.795832,7.211103,3.872983,7.211103,7.000000,3.162278,3.162278,5.567764,3.872983
4,6.480741,4.123106,5.196152,3.162278,3.316625,4.795832,3.316625,3.162278,5.385165,5.830952,...,5.656854,5.000000,7.483315,4.000000,7.211103,7.000000,3.316625,3.316625,5.567764,4.242641
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,11.180340,10.392305,11.090537,11.090537,11.224972,11.224972,11.357817,11.090537,11.618950,11.180340,...,11.532563,10.488088,11.357817,10.723805,10.908712,10.862780,10.723805,10.723805,11.180340,11.045361
2206,11.180340,10.440307,11.180340,11.180340,11.313708,11.224972,11.445523,11.180340,11.618950,11.180340,...,11.532563,10.535654,11.357817,10.723805,10.908712,10.954451,10.723805,10.723805,11.180340,11.090537
2207,11.180340,10.440307,11.180340,11.180340,11.313708,11.224972,11.445523,11.180340,11.618950,11.180340,...,11.575837,10.535654,11.357817,10.770330,11.045361,10.954451,10.723805,10.723805,11.180340,11.090537
2208,11.180340,10.440307,11.489125,11.224972,11.357817,11.269428,11.489125,11.224972,11.618950,11.224972,...,11.618950,10.535654,11.401754,10.816654,11.135529,10.954451,10.723805,10.723805,11.224972,11.224972


In [59]:
similarity= neighbors_k

In [60]:
Dmean=np.mean(similarity[1,:])

In [61]:
round(Dmean, 2)

np.float64(3.37)

In [62]:
std=np.std(similarity[1,:])

In [63]:
round(std, 2)

np.float64(1.53)

In [64]:
model_AD_limit=Dmean+std*0.5
print(np.round(model_AD_limit, 2))

4.13


In [65]:
neighbors_k_ts= pairwise_distances(desc_ws,Y=desc_ts, n_jobs=-1)
neighbors_k_ts.sort(0)

In [66]:
x_ts_AD=pd.DataFrame(neighbors_k_ts)
x_ts_AD

,0,1,2,3,4,5,6,7,8,9,...,543,544,545,546,547,548,549,550,551,552
0,5.000000,5.291503,3.605551,4.582576,3.741657,4.690416,3.605551,3.316625,4.242641,3.464102,...,5.099020,3.162278,4.242641,2.449490,2.645751,3.741657,3.162278,0.000000,1.732051,4.123106
1,5.099020,5.744563,4.472136,5.744563,3.741657,4.690416,3.605551,4.472136,4.472136,4.000000,...,5.196152,3.741657,4.242641,2.449490,2.828427,4.242641,3.162278,0.000000,3.464102,4.472136
2,5.099020,6.000000,4.582576,5.744563,4.242641,4.898979,3.872983,4.690416,5.196152,4.123106,...,5.196152,4.000000,4.690416,3.316625,2.828427,4.242641,3.464102,1.414214,3.464102,4.472136
3,5.196152,6.082763,5.000000,5.744563,4.358899,5.000000,4.242641,5.291503,5.196152,4.242641,...,5.385165,4.123106,4.690416,3.741657,3.162278,4.472136,3.872983,3.000000,3.872983,4.472136
4,5.291503,6.164414,5.099020,5.830952,4.358899,5.196152,4.358899,5.291503,5.291503,4.242641,...,5.656854,4.472136,4.795832,3.741657,3.464102,4.690416,4.123106,3.605551,3.872983,4.795832
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,11.357817,11.045361,11.045361,11.135529,11.000000,11.269428,10.862780,11.661904,11.224972,11.445523,...,11.532563,10.440307,10.535654,10.148892,10.440307,10.344080,10.677078,10.488088,10.246951,11.575837
2206,11.357817,11.045361,11.090537,11.180340,11.090537,11.269428,10.862780,11.661904,11.224972,11.445523,...,11.532563,10.440307,10.535654,10.198039,10.488088,10.344080,10.677078,10.488088,10.246951,11.618950
2207,11.357817,11.090537,11.090537,11.180340,11.090537,11.269428,10.862780,11.661904,11.224972,11.445523,...,11.575837,10.488088,10.583005,10.198039,10.488088,10.392305,10.723805,10.535654,10.246951,11.618950
2208,11.445523,11.090537,11.135529,11.180340,11.313708,11.269428,10.908712,11.874342,11.704700,11.747340,...,11.575837,10.488088,10.583005,10.246951,10.488088,10.440307,10.770330,10.583005,10.246951,11.747340


In [67]:
similarity_ts= neighbors_k_ts
cpd_AD=similarity_ts[0,:]
cpd_value = np.round(cpd_AD, 3)
print(cpd_value)

[5.    5.292 3.606 4.583 3.742 4.69  3.606 3.317 4.243 3.464 5.099 3.162
 2.828 3.    4.    3.162 3.464 3.464 2.449 4.796 0.    1.414 3.317 4.472
 3.317 1.732 3.606 3.873 3.464 3.464 3.464 4.583 3.    4.    3.    4.359
 2.646 3.873 5.    4.69  5.099 5.657 2.646 0.    3.742 2.828 3.464 3.742
 2.449 4.    3.162 4.472 3.317 2.646 1.414 3.742 4.    5.385 0.    0.
 3.162 5.477 3.873 3.162 3.162 4.    3.    4.899 3.162 2.    4.243 3.873
 0.    4.    2.828 0.    6.    3.162 3.873 2.646 3.    1.    4.    3.464
 4.359 3.742 4.    0.    3.873 4.583 3.    3.464 4.243 3.    4.    3.873
 4.899 4.243 0.    3.742 3.873 3.162 3.    2.828 3.742 3.742 3.742 4.359
 1.    1.732 3.464 4.243 3.742 4.359 0.    4.359 3.    3.162 3.317 3.606
 3.742 0.    2.449 5.196 0.    2.828 1.    3.162 3.742 4.359 4.583 4.69
 4.69  6.708 3.742 3.742 1.414 3.606 1.    3.873 4.359 2.    3.742 3.317
 3.317 1.    3.    3.873 3.    1.    0.    3.317 1.    3.317 3.162 1.
 8.246 4.69  4.69  3.162 3.162 1.414 4.359 4.472 3.873 3. 

In [68]:
cpd_AD = np.where(cpd_value <= model_AD_limit, True, False)
print(cpd_AD)

[False False  True False  True False  True  True False  True False  True
  True  True  True  True  True  True  True False  True  True  True False
  True  True  True  True  True  True  True False  True  True  True False
  True  True False False False False  True  True  True  True  True  True
  True  True  True False  True  True  True  True  True False  True  True
  True False  True  True  True  True  True False  True  True False  True
  True  True  True  True False  True  True  True  True  True  True  True
 False  True  True  True  True False  True  True False  True  True  True
 False False  True  True  True  True  True  True  True  True  True False
  True  True  True False  True False  True False  True  True  True  True
  True  True  True False  True  True  True  True  True False False False
 False False  True  True  True  True  True  True False  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
 False False False  True  True  True False False  T

In [69]:
print("Coverage = ", round(sum(cpd_AD) / len(cpd_AD), 2))

Coverage =  0.77


In [70]:
print("Indices of substances included in AD = ", np.where(cpd_AD != 0)[0])

Indices of substances included in AD =  [  2   4   6   7   9  11  12  13  14  15  16  17  18  20  21  22  24  25
  26  27  28  29  30  32  33  34  36  37  42  43  44  45  46  47  48  49
  50  52  53  54  55  56  58  59  60  62  63  64  65  66  68  69  71  72
  73  74  75  77  78  79  80  81  82  83  85  86  87  88  90  91  93  94
  95  98  99 100 101 102 103 104 105 106 108 109 110 112 114 116 117 118
 119 120 121 122 124 125 126 127 128 134 135 136 137 138 139 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 159 160 161 164 165 166
 169 170 171 172 173 174 175 176 178 179 180 181 182 183 184 186 187 191
 193 194 195 199 202 203 205 206 208 209 210 211 213 214 215 216 217 218
 221 222 224 225 226 228 229 230 231 233 234 235 236 237 238 239 240 241
 242 243 244 245 246 247 248 249 250 251 252 253 255 256 258 259 261 262
 263 265 266 267 268 269 270 273 274 276 277 278 279 282 283 284 286 287
 288 289 290 291 293 295 296 298 299 300 301 304 305 307 308 309 310 311
 312 313 31

In [71]:
out_Ad=list(np.where(cpd_AD == 0)[0])

# Prediction only for molecules included in  AD

In [72]:
y_pred_MLPR_ad=list(y_pred_MLPR)

In [73]:
y_pred_MLPR_ad[:] = [x for i,x in enumerate(y_pred_MLPR_ad) if i not in out_Ad]

In [74]:
len(y_pred_MLPR_ad)

424

In [75]:
y_ts_ad=list(y_ts)

In [76]:
y_ts_ad[:] = [x for i,x in enumerate(y_ts_ad) if i not in out_Ad]

In [77]:
len(y_ts_ad)

424

In [78]:
Q2_TS = round(r2_score(y_ts_ad, y_pred_MLPR_ad), 2)
Q2_TS

0.53

In [79]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts_ad, y_pred_MLPR_ad)), 2))
RMSE_TS

0.58

# k-nearest neighbors

In [81]:
k_range = list(range(1, 31))
param_grid = dict(n_neighbors=k_range)

In [82]:
m = GridSearchCV(KNeighborsRegressor(), param_grid, n_jobs=-1, cv=cv, verbose=1)

In [83]:
m.fit(desc_ws, y_tr)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,KNeighborsRegressor()
,param_grid,"{'n_neighbors': [1, 2, ...]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,3


In [84]:
best_kNN = m.best_estimator_

In [85]:
m.best_params_

{'n_neighbors': 3}

In [87]:
y_pred_ws_kNN = best_kNN.predict(desc_ws)

In [88]:
R2_WS = round(r2_score(y_tr, y_pred_ws_kNN), 2)
R2_WS

0.79

In [90]:
RMSE_WS=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_ws_kNN)), 2))
RMSE_WS

0.39

In [91]:
y_pred_CV_kNN = cross_val_predict(best_kNN, desc_ws, y_tr, cv=cv)

In [92]:
y_pred_CV_kNN

array([4.9666667, 5.6766667, 4.793333 , ..., 8.026668 , 6.2933335,
       7.65     ], shape=(2210,), dtype=float32)

In [93]:
Q2_CV = round(r2_score(y_tr, y_pred_CV_kNN), 2)
Q2_CV

0.52

In [95]:
RMSE_CV=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_CV_kNN)), 2))
RMSE_CV

0.59

# 9. Prediction for test set's molecules

In [18]:
x_ts = np.array(desc_ws, dtype=np.float32)
y_ts = np.array(y_ts, dtype=np.float32)

In [19]:
y_pred_kNN = best_kNN.predict(desc_ts)

In [20]:
Q2_TS = round(r2_score(y_ts, y_pred_kNN), 2)
Q2_TS

0.53

In [21]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts, y_pred_kNN)), 2))
RMSE_TS

0.58

# save the model to disk

In [101]:
pickle.dump(best_kNN, open('models/ECFP4/ECFP4_kNN.pkl', 'wb'))

# load the model from disk

In [17]:
best_kNN = pickle.load(open('models/ECFP4/ECFP4_kNN.pkl', 'rb'))

#  Estimating applicability domain. Method - Euclidian distances, K=1

In [22]:
neighbors_k= pairwise_distances(desc_ws, n_jobs=-1)
neighbors_k.sort(0)

In [23]:
df_tr=pd.DataFrame(neighbors_k)
df_tr

,0,1,2,3,4,5,6,7,8,9,...,2200,2201,2202,2203,2204,2205,2206,2207,2208,2209
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,5.916080,3.162278,3.872983,0.000000,2.645751,3.162278,3.162278,0.000000,3.741657,4.123106,...,3.872983,3.464102,7.000000,3.464102,4.898979,6.928203,0.000000,0.000000,4.795832,3.316625
2,6.000000,3.741657,5.196152,2.645751,2.645751,4.242641,3.162278,2.645751,5.291503,4.795832,...,4.472136,4.242641,7.071068,3.605551,6.708204,7.000000,3.000000,3.000000,5.291503,3.605551
3,6.403124,4.000000,5.196152,2.828427,3.000000,4.472136,3.162278,2.828427,5.291503,5.830952,...,5.000000,4.795832,7.211103,3.872983,7.211103,7.000000,3.162278,3.162278,5.567764,3.872983
4,6.480741,4.123106,5.196152,3.162278,3.316625,4.795832,3.316625,3.162278,5.385165,5.830952,...,5.656854,5.000000,7.483315,4.000000,7.211103,7.000000,3.316625,3.316625,5.567764,4.242641
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,11.180340,10.392305,11.090537,11.090537,11.224972,11.224972,11.357817,11.090537,11.618950,11.180340,...,11.532563,10.488088,11.357817,10.723805,10.908712,10.862780,10.723805,10.723805,11.180340,11.045361
2206,11.180340,10.440307,11.180340,11.180340,11.313708,11.224972,11.445523,11.180340,11.618950,11.180340,...,11.532563,10.535654,11.357817,10.723805,10.908712,10.954451,10.723805,10.723805,11.180340,11.090537
2207,11.180340,10.440307,11.180340,11.180340,11.313708,11.224972,11.445523,11.180340,11.618950,11.180340,...,11.575837,10.535654,11.357817,10.770330,11.045361,10.954451,10.723805,10.723805,11.180340,11.090537
2208,11.180340,10.440307,11.489125,11.224972,11.357817,11.269428,11.489125,11.224972,11.618950,11.224972,...,11.618950,10.535654,11.401754,10.816654,11.135529,10.954451,10.723805,10.723805,11.224972,11.224972


In [24]:
similarity= neighbors_k

In [25]:
Dmean=np.mean(similarity[1,:])

In [26]:
round(Dmean, 2)

np.float64(3.37)

In [27]:
std=np.std(similarity[1,:])

In [28]:
round(std, 2)

np.float64(1.53)

In [29]:
model_AD_limit=Dmean+std*0.5
print(np.round(model_AD_limit, 2))

4.13


In [30]:
neighbors_k_ts= pairwise_distances(desc_ws,Y=desc_ts, n_jobs=-1)
neighbors_k_ts.sort(0)

In [31]:
x_ts_AD=pd.DataFrame(neighbors_k_ts)
x_ts_AD

,0,1,2,3,4,5,6,7,8,9,...,543,544,545,546,547,548,549,550,551,552
0,5.000000,5.291503,3.605551,4.582576,3.741657,4.690416,3.605551,3.316625,4.242641,3.464102,...,5.099020,3.162278,4.242641,2.449490,2.645751,3.741657,3.162278,0.000000,1.732051,4.123106
1,5.099020,5.744563,4.472136,5.744563,3.741657,4.690416,3.605551,4.472136,4.472136,4.000000,...,5.196152,3.741657,4.242641,2.449490,2.828427,4.242641,3.162278,0.000000,3.464102,4.472136
2,5.099020,6.000000,4.582576,5.744563,4.242641,4.898979,3.872983,4.690416,5.196152,4.123106,...,5.196152,4.000000,4.690416,3.316625,2.828427,4.242641,3.464102,1.414214,3.464102,4.472136
3,5.196152,6.082763,5.000000,5.744563,4.358899,5.000000,4.242641,5.291503,5.196152,4.242641,...,5.385165,4.123106,4.690416,3.741657,3.162278,4.472136,3.872983,3.000000,3.872983,4.472136
4,5.291503,6.164414,5.099020,5.830952,4.358899,5.196152,4.358899,5.291503,5.291503,4.242641,...,5.656854,4.472136,4.795832,3.741657,3.464102,4.690416,4.123106,3.605551,3.872983,4.795832
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,11.357817,11.045361,11.045361,11.135529,11.000000,11.269428,10.862780,11.661904,11.224972,11.445523,...,11.532563,10.440307,10.535654,10.148892,10.440307,10.344080,10.677078,10.488088,10.246951,11.575837
2206,11.357817,11.045361,11.090537,11.180340,11.090537,11.269428,10.862780,11.661904,11.224972,11.445523,...,11.532563,10.440307,10.535654,10.198039,10.488088,10.344080,10.677078,10.488088,10.246951,11.618950
2207,11.357817,11.090537,11.090537,11.180340,11.090537,11.269428,10.862780,11.661904,11.224972,11.445523,...,11.575837,10.488088,10.583005,10.198039,10.488088,10.392305,10.723805,10.535654,10.246951,11.618950
2208,11.445523,11.090537,11.135529,11.180340,11.313708,11.269428,10.908712,11.874342,11.704700,11.747340,...,11.575837,10.488088,10.583005,10.246951,10.488088,10.440307,10.770330,10.583005,10.246951,11.747340


In [32]:
similarity_ts= neighbors_k_ts
cpd_AD=similarity_ts[0,:]
cpd_value = np.round(cpd_AD, 3)
print(cpd_value)

[5.    5.292 3.606 4.583 3.742 4.69  3.606 3.317 4.243 3.464 5.099 3.162
 2.828 3.    4.    3.162 3.464 3.464 2.449 4.796 0.    1.414 3.317 4.472
 3.317 1.732 3.606 3.873 3.464 3.464 3.464 4.583 3.    4.    3.    4.359
 2.646 3.873 5.    4.69  5.099 5.657 2.646 0.    3.742 2.828 3.464 3.742
 2.449 4.    3.162 4.472 3.317 2.646 1.414 3.742 4.    5.385 0.    0.
 3.162 5.477 3.873 3.162 3.162 4.    3.    4.899 3.162 2.    4.243 3.873
 0.    4.    2.828 0.    6.    3.162 3.873 2.646 3.    1.    4.    3.464
 4.359 3.742 4.    0.    3.873 4.583 3.    3.464 4.243 3.    4.    3.873
 4.899 4.243 0.    3.742 3.873 3.162 3.    2.828 3.742 3.742 3.742 4.359
 1.    1.732 3.464 4.243 3.742 4.359 0.    4.359 3.    3.162 3.317 3.606
 3.742 0.    2.449 5.196 0.    2.828 1.    3.162 3.742 4.359 4.583 4.69
 4.69  6.708 3.742 3.742 1.414 3.606 1.    3.873 4.359 2.    3.742 3.317
 3.317 1.    3.    3.873 3.    1.    0.    3.317 1.    3.317 3.162 1.
 8.246 4.69  4.69  3.162 3.162 1.414 4.359 4.472 3.873 3. 

In [33]:
cpd_AD = np.where(cpd_value <= model_AD_limit, True, False)
print(cpd_AD)

[False False  True False  True False  True  True False  True False  True
  True  True  True  True  True  True  True False  True  True  True False
  True  True  True  True  True  True  True False  True  True  True False
  True  True False False False False  True  True  True  True  True  True
  True  True  True False  True  True  True  True  True False  True  True
  True False  True  True  True  True  True False  True  True False  True
  True  True  True  True False  True  True  True  True  True  True  True
 False  True  True  True  True False  True  True False  True  True  True
 False False  True  True  True  True  True  True  True  True  True False
  True  True  True False  True False  True False  True  True  True  True
  True  True  True False  True  True  True  True  True False False False
 False False  True  True  True  True  True  True False  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
 False False False  True  True  True False False  T

In [34]:
print("Coverage = ", round(sum(cpd_AD) / len(cpd_AD), 2))

Coverage =  0.77


In [35]:
print("Indices of substances included in AD = ", np.where(cpd_AD != 0)[0])

Indices of substances included in AD =  [  2   4   6   7   9  11  12  13  14  15  16  17  18  20  21  22  24  25
  26  27  28  29  30  32  33  34  36  37  42  43  44  45  46  47  48  49
  50  52  53  54  55  56  58  59  60  62  63  64  65  66  68  69  71  72
  73  74  75  77  78  79  80  81  82  83  85  86  87  88  90  91  93  94
  95  98  99 100 101 102 103 104 105 106 108 109 110 112 114 116 117 118
 119 120 121 122 124 125 126 127 128 134 135 136 137 138 139 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 159 160 161 164 165 166
 169 170 171 172 173 174 175 176 178 179 180 181 182 183 184 186 187 191
 193 194 195 199 202 203 205 206 208 209 210 211 213 214 215 216 217 218
 221 222 224 225 226 228 229 230 231 233 234 235 236 237 238 239 240 241
 242 243 244 245 246 247 248 249 250 251 252 253 255 256 258 259 261 262
 263 265 266 267 268 269 270 273 274 276 277 278 279 282 283 284 286 287
 288 289 290 291 293 295 296 298 299 300 301 304 305 307 308 309 310 311
 312 313 31

In [36]:
out_Ad=list(np.where(cpd_AD == 0)[0])

# Prediction only for molecules included in  AD

In [37]:
y_pred_kNN_ad=list(y_pred_kNN)

In [38]:
y_pred_kNN_ad[:] = [x for i,x in enumerate(y_pred_kNN_ad) if i not in out_Ad]

In [39]:
len(y_pred_kNN_ad)

424

In [40]:
y_ts_ad=list(y_ts)

In [41]:
y_ts_ad[:] = [x for i,x in enumerate(y_ts_ad) if i not in out_Ad]

In [42]:
len(y_ts_ad)

424

In [43]:
Q2_TS = round(r2_score(y_ts_ad, y_pred_kNN_ad), 2)
Q2_TS

0.58

In [44]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts_ad, y_pred_kNN_ad)), 2))
RMSE_TS

0.55